In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences
from dataloaders._ts_dataloader      import DataLoaderFactory, FullSeriesDataset
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common._base_model import BaseModel
from common.train import train, train_distributed, eval_test
from models.linear import TinyLinearModel

warnings.filterwarnings("ignore")
print("imports OK")

In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    context_length:    int  = 64,
    horizon:           int  = 6,
    n_channels:        int  = 3,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    normalize:         bool = False,
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        context_length         = context_length,
        horizon                = horizon,
        n_channels             = n_channels,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        normalize              = normalize,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
        loss                   = 'mae',
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

## Test — `train()`, resume from checkpoint

In [ ]:
print("=" * 60)
print("TEST — resume training from checkpoint")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=10, checkpoint_dir=tmpdir,
    )
    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()
    model = TinyLinearModel(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"))
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=step_after_first + 10, checkpoint_dir=tmpdir,
    )
    factory2     = DataLoaderFactory(mcfg2, dcfg)
    model2       = TinyLinearModel(mcfg2)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{tmpdir}/final.pt",
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"

print("\n✓ TEST PASSED")

## Test — `train_distributed()`, sharded dataset, mp.spawn

In [ ]:
print("=" * 60)
print("TEST — train_distributed (mp.spawn, 2 ranks, sharded)")
print("=" * 60)

# Use gloo backend so this works on CPU / machines without NCCL
BACKEND    = "nccl" if torch.cuda.is_available() else "gloo"
WORLD_SIZE = min(2, torch.cuda.device_count() or 2)
print(f"backend={BACKEND}  world_size={WORLD_SIZE}")

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    csv_path  = f"{tmpdir}/data.csv"

    CTX     = 32
    HORIZON = 6

    df = make_df(n_series=3, n_steps=600, seed=99)
    df.to_csv(csv_path, index=False)

    write_sharded_dataset(
        df             = df,
        out_dir        = shard_dir,
        val_size       = 80,
        test_size      = 80,
        context_length = CTX,
        shard_size     = 100,   # small shards — ensures each rank gets ≥1 shard
    )

    shard_files = sorted(Path(shard_dir).glob("shard_*.parquet"))
    print(f"{len(shard_files)} train shards written")
    assert len(shard_files) >= WORLD_SIZE, \
        f"need ≥{WORLD_SIZE} shards for {WORLD_SIZE} ranks, got {len(shard_files)}"

    mcfg = make_mcfg(
        context_length  = CTX,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        val_check_interval = 10,
        checkpoint_dir  = tmpdir,
    )
    entry = make_entry(
        path        = csv_path,
        name        = "dist_ds",
        horizon     = HORIZON,
        val_size    = 80,
        test_size   = 80,
        sharded_dir = shard_dir,
    )
    dcfg = make_dcfg([entry])

    # Factory is created BEFORE spawn — shared config, datasets rebuilt per rank
    factory = DataLoaderFactory(mcfg, dcfg)
    model   = TinyLinearModel(mcfg)

    train_distributed(
        model      = model,
        mcfg       = mcfg,
        factory    = factory,
        backend    = BACKEND,
        use_spawn  = True,
        world_size = WORLD_SIZE,
        seed       = 42,
    )

    assert Path(tmpdir, "final.pt").exists(), "final.pt not saved by rank 0"
    print("final.pt saved ✓")

    # Load weights and run test on single GPU
    test_model = TinyLinearModel(mcfg)
    factory_test = DataLoaderFactory(mcfg, dcfg)
    BaseModel.load_weights(f"{tmpdir}/final.pt", test_model)
    test_model.setup_inference(mcfg, device=torch.device("cpu"))
    preds = eval_test(test_model, factory_test, device=torch.device("cpu"))

    assert "dist_ds" in preds, f"missing keys in preds: {preds.keys()}"
    
print("\n✓ TEST PASSED")